# OBJECTIVES
This file demonstrates improvisation of Retrieval in RAG pipeline with focus on retrieval using query expansion technique.

# SETUP AND IMPORTS

In [1]:
import chromadb
from langchain_chroma import Chroma

import os
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader


c:\Users\Taniya Gupta\Desktop\Assignments\AG0856_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found"
    )

In [3]:
from groq import Groq

client = Groq()

## Loading tesla reports

In [4]:
pdf_folder_location = "tesla-annual-reports"

In [5]:
pdf_loader = PyPDFDirectoryLoader(pdf_folder_location)

## Chunking

In [6]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=1000,
    chunk_overlap=150
)

In [7]:
tesla_10k_chunks = pdf_loader.load_and_split(text_splitter)

In [8]:
len(tesla_10k_chunks)

2059

## Embeddings

In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [11]:

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Taniya Gupta\AppData\Local\Temp\ipykernel_7356\2427895705.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")


In [9]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'

## Vector storage

In [12]:
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


In [13]:
vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [14]:
chromadb_client.count_collections()

1

In [15]:
batch_size = 500

for start_idx in range(0, len(tesla_10k_chunks), batch_size):

    batch_docs = tesla_10k_chunks[start_idx:start_idx + batch_size]

    vectorstore.add_documents(
        documents=batch_docs,
        ids=[
            f"text_{idx}"
            for idx in range(start_idx, start_idx + len(batch_docs))
        ]
    )

In [16]:
collection = chromadb_client.get_collection(tesla_10k_collection)

In [17]:
collection.count()

2059

In [18]:
model_name = 'openai/gpt-oss-120b'

In [20]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}
)

# USING QUERY EXPANSION

In [21]:
query_expansion_system_message = """
You are a financial domain expert assisting with Tesla 10-K analysis.

Generate 4 query variations from different perspectives:
1. Financial analyst perspective
2. Risk factor perspective
3. Operational perspective
4. Synonym/alternative terminology perspective

Preserve the original intent.
Return only the queries.
One query per line.
"""
user_message_template="""
<Question>
{question}
</Question>
"""

In [22]:
user_input = "Any specific fines levied on the company in 2022?"

In [23]:
prompt=[
    {"role": "system", "content": query_expansion_system_message},
    {"role": "user", "content": user_message_template.format(question=user_input)}
]

In [24]:
query_expansions=client.chat.completions.create(
    model=model_name,
    messages=prompt,
    temperature=0,
)

In [25]:
print(query_expansions.choices[0].message.content)

What fines were imposed on Tesla in 2022 and what was their impact on the company's financial statements?  
What regulatory fines did Tesla incur in 2022 that could affect its risk profile?  
Which operational penalties or fines were levied against Tesla in 2022?  
Were any penalties or sanctions assessed against Tesla in 2022?


In [26]:
query_expansions_list = query_expansions.choices[0].message.content.strip().split("\n")

In [27]:
query_expansions_list

["What fines were imposed on Tesla in 2022 and what was their impact on the company's financial statements?  ",
 'What regulatory fines did Tesla incur in 2022 that could affect its risk profile?  ',
 'Which operational penalties or fines were levied against Tesla in 2022?  ',
 'Were any penalties or sanctions assessed against Tesla in 2022?']

In [28]:
expanded_context_list = []


In [29]:
for query in query_expansions_list:
    expanded_context_list.extend([d.page_content for d in retriever.invoke(query)])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [30]:
import re

def clean_text(text):
    # Replace tabs/newlines with spaces
    text = text.replace("\t", " ")
    text = text.replace("\n", " ")

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [31]:
expanded_context_list = [
    clean_text(doc)
    for doc in expanded_context_list
]

In [32]:
expanded_context_list

['penalties could include ongoing audit requirements, data protection authority investigations, legal proceedings by international governmental entities or others resulting in mandated disclosure of sensitive data or other commercially unfavorable terms. Notwithstanding our efforts to protect the security and integrity of our customers’ personal information, we may be required to expend significant resources to comply with data breach requirements if, for example, third parties improperly obtain and use the personal information of our customers or we otherwise experience a data loss with respect to customers’ personal information. A major breach of our network security and systems may result in fines, penalties and damages and harm our brand, prospects and operating results. We could be subject to liability, penalties and other restrictive sanctions and adverse consequences arising out of certain governmental investigations and proceedings. We are cooperating with certain government in

In [33]:
final_context_documents = set(expanded_context_list)

In [34]:
final_context_documents

{'Northern District of California. On February 27, 2023, a proposed class action was filed in the U.S. District Court for the Northern District of California against Tesla, Inc., Elon Musk and certain current and former Company executives. The complaint alleges that the defendants made material misrepresentations and omissions about the Company’s Autopilot and FSD Capability technologies and seeks money damages and other relief on behalf of persons who purchased Tesla stock between February 19, 2019 and February 17, 2023. An amended complaint was filed on September 5, 2023, naming only Tesla, Inc. and Elon Musk as defendants. On November 6, 2023, Tesla moved to dismiss the amended complaint. On March 14, 2023, a proposed class action was filed against Tesla, Inc. in the U.S. District Court for the Northern District of California. Several similar complaints have also been filed in the same court and these cases have now all been consolidated. These complaints allege that Tesla violates 

In [35]:
final_context_documents= list(final_context_documents)
final_context_documents

['enrichment, and violation of the federal securities laws in connection with alleged race and gender discrimination and sexual harassment. Among other things, plaintiffs seek declaratory and injunctive relief, unspecified damages payable to Tesla, and attorneys’ fees. On July 22, 2022, the Court consolidated the two cases and on September 6, 2022, plaintiffs filed a consolidated complaint. On November 7, 2022, the defendants filed a motion to dismiss the case. Plaintiffs filed a response of January 13, 2023, and the defendants’ reply is due February 17, 2023. Certain Investigations and Other Matters We receive requests for information from regulators and governmental authorities, such as the National Highway Traffic Safety Administration, the National Transportation Safety Board, the SEC, the Department of Justice (“DOJ”) and various state, federal, and international agencies. We routinely cooperate with such regulatory and governmental requests, including subpoenas, formal and inform

In [36]:
system_message = """
    You are an assistant to a financial services firm who answers user queries on annual reports.
    User input will have the context required by you to answer user queries.
    This context will be delimited by: <Context> and </Context>.
    The context contains references to specific portions of a document relevant to the user query.

    User queries will be delimited by: <Question> and </Question>.

    Please answer user queries only using the context provided in the input.
    Do not mention anything about the context in your final answer. Your response should only contain the answer to the question.

    If the answer is not found in the context, respond "I don't know".
    """

message_template = """
    <Context>
    Here are some documents that are relevant to the question mentioned below.
    {context}
    </Context>

    <Question>
    {question}
    </Question>
    """

prompt=[
        {"role": "system", "content": system_message},  
        {"role": "user", "content": message_template.format(context="\n".join(final_context_documents), question=user_input)}

    ]
response=client.chat.completions.create(
        model=model_name,   
        messages=prompt,
        temperature=0,
    )
answer = response.choices[0].message.content
print(answer)

Yes. In 2022 a fine of €600,000 was levied on the company, which resulted from a settlement following a hearing on November 24, 2022.


# Evaluation and analysis

In [37]:

benchmark_questions = {
"Q1":"Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure?",
"Q2":"Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures.",
"Q3":"Assess Tesla's concentration risk across factories, suppliers, raw materials and geographies.",
"Q4":"Compare the strategic importance of automotive operations and energy generation/storage."
}


In [38]:

def baseline_retrieval(question):
    return retriever.invoke(question)

def expanded_retrieval(question):

    prompt=[
        {"role":"system","content":query_expansion_system_message},
        {"role":"user","content":user_message_template.format(question=question)}
    ]

    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    expanded_queries = response.choices[0].message.content.strip().split("\n")

    all_docs = []
    retrieved_by = {}

    for query in expanded_queries:
        docs = retriever.invoke(query)

        for doc in docs:

            chunk_id = doc.metadata.get("id","unknown")

            retrieved_by.setdefault(chunk_id, []).append(query)
            all_docs.append(doc)

    unique_docs = []

    seen = set()

    for doc in all_docs:
        text = doc.page_content

        if text not in seen:
            seen.add(text)
            unique_docs.append(doc)

    return expanded_queries, unique_docs[:10], retrieved_by


In [39]:

import json

evaluation_results = []

for qid, question in benchmark_questions.items():

    baseline_docs = baseline_retrieval(question)

    expanded_queries, expanded_docs, retrieved_by = expanded_retrieval(question)

    evaluation_results.append({
        "question_id": qid,
        "original_query": question,
        "expanded_queries": expanded_queries,
        "baseline_top_chunks": [d.page_content[:120] for d in baseline_docs],
        "expanded_top_chunks": [d.page_content[:120] for d in expanded_docs],
        "retrieved_by": retrieved_by,
        "retrieval_improvement_analysis":
        "Expanded retrieval increased coverage by searching alternative phrasings."
    })

with open("query_expansion_evaluation.json","w") as f:
    json.dump(evaluation_results, f, indent=4)



## Comparative Analysis

| Question | Baseline | Expanded |
|-----------|-----------|-----------|
| Q1 | Medium | High |
| Q2 | Medium | High |
| Q3 | Low | High |
| Q4 | Medium | High |

